# Notebook 11 — Table 8: Cluster Performance Under Macro Regime Switches

**Data needed:**
- `data/grid/cluster_month_panel_K_50_lambda_1000000.csv`
- `data/factor_returns.csv`   (for FF5 GRS test)
- `data/macro_predictors.csv` (columns: dp, dy, ep, tbl, tms, dfy, infl, svar, lev, ni)

**Output:** `Table8_Macro_Regime_Switches.{csv,tex}`

For each macro variable × regime (Top/Btm/Combined):
- AVG: mean monthly excess return of tangency portfolio
- SR: annualised Sharpe ratio
- GRS: Gibbons-Ross-Shanken F-statistic vs FF5
- p-value, mean|α|, √mean(α²), mean R²

In [5]:
import os, sys, warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
warnings.filterwarnings("ignore")

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, REPO_ROOT)

from utils.data_utils import (load_cluster_panel, load_cluster_ranking,
                               pivot_and_rank, nw_tstat, stars,
                               ann_sharpe, save_table)
from utils.portfolio_utils import estimate_hj_sdf

NW_LAGS  = 3
ROLL_WIN = 120    # 10-year rolling window

# ── Paths ──────────────────────────────────────────────────────────
BASE       = "/ssd1/songjiangliu/shared/asset_clustering"
DATA_PATH  = os.path.join(BASE, "Data")
MACRO_FILE = os.path.join(DATA_PATH,
             "Macro_economic_PredictorData2021_Amit_goyal.xlsx")
FACTOR_CSV = os.path.join(DATA_PATH, "All_factors_JFE.csv")

FF5_COLS = ["MKT", "SMB", "HML", "RMW", "CMA"]

print("Setup complete.")
print(f"Macro file  : {MACRO_FILE}")
print(f"Factor file : {FACTOR_CSV}")

Setup complete.
Macro file  : /ssd1/songjiangliu/shared/asset_clustering/Data/Macro_economic_PredictorData2021_Amit_goyal.xlsx
Factor file : /ssd1/songjiangliu/shared/asset_clustering/Data/All_factors_JFE.csv


In [4]:
# ── Load all data ─────────────────────────────────────────────────

ranking        = load_cluster_ranking()
df          = load_cluster_panel(K=50, lam=1_000_000)
cr   =     (df, dropna=True)
cr_r, rank_map = pivot_and_rank(df_cl, lam=1_000_000, ranking_df=ranking)

try:
    factors = load_factor_data()
    ff5_avail = [c for c in FF5_COLS if c in factors.columns]
    print(f"Factors loaded. FF5 cols: {ff5_avail}")
except FileNotFoundError:
    factors = pd.DataFrame(index=cr_r.index); ff5_avail = []
    print("factor_returns.csv not found — GRS will be NaN")

try:
    macro = load_macro_data()
    avail_macro = {k: v for k, v in MACRO_VARS.items() if v in macro.columns}
    print(f"Macro loaded: {list(avail_macro.keys())}")
except FileNotFoundError:
    macro = pd.DataFrame(index=cr_r.index); avail_macro = {}
    print("macro_predictors.csv not found — Table 8 will be empty")

# Align
common = cr_r.index.intersection(factors.index).intersection(macro.index)          if len(factors.columns) > 0 and len(macro.columns) > 0          else cr_r.index
cr_c = cr_r.loc[cr_r.index.intersection(common)]
fd_c = factors.loc[factors.index.intersection(common)] if len(factors.columns)>0        else pd.DataFrame(index=common)
mc_c = macro.loc[macro.index.intersection(common)]  if len(macro.columns)>0        else pd.DataFrame(index=common)
print(f"Common period: {common[0].date()} → {common[-1].date()} ({len(common)}m)")

SyntaxError: invalid syntax (3734340065.py, line 5)

In [3]:
# ── GRS test helper ───────────────────────────────────────────────
def grs_test(cr_sub, fd_sub, ff5_cols, nw_lags=3):
    if len(ff5_cols) == 0 or len(cr_sub) < 30:
        return np.nan, np.nan, [], []
    idx = cr_sub.index.intersection(fd_sub.index)
    R = cr_sub.loc[idx].values
    F = fd_sub[ff5_cols].loc[idx].dropna().values
    # Align after dropna
    min_len = min(R.shape[0], F.shape[0])
    R = R[:min_len]; F = F[:min_len]
    T, N = R.shape; K = F.shape[1]
    X = np.column_stack([np.ones(T), F])
    B = np.linalg.lstsq(X, R, rcond=None)[0]
    alp = B[0]
    E   = R - X @ B
    Sig_e = (E.T @ E) / max(T - K - 1, 1)
    mu_f  = F.mean(axis=0)
    Sig_f = np.cov(F, rowvar=False) + 1e-6*np.eye(K)
    try:
        adj   = 1 + mu_f @ np.linalg.solve(Sig_f, mu_f)
        grs_f = ((T - N - K) / N) * (1/adj) * (alp @ np.linalg.solve(Sig_e, alp))
        p_val = 1 - stats.f.cdf(grs_f, N, T - N - K)
    except Exception:
        grs_f, p_val = np.nan, np.nan
    # Alpha diagnostics
    alphas, r2s = [], []
    for i in range(N):
        res = sm.OLS(R[:,i], X).fit(cov_type="HAC", cov_kwds={"maxlags":nw_lags})
        alphas.append(res.params[0] * 100)
        r2s.append(res.rsquared * 100)
    return float(grs_f), float(p_val), alphas, r2s

In [4]:
# ── Build Table 8 ─────────────────────────────────────────────────
rows = []

for paper_name, col in avail_macro.items():
    print(f"  {paper_name}...", end=" ", flush=True)
    macro_series = mc_c[col].dropna()
    
    # Rolling percentile regime
    thresh = macro_series.rolling(ROLL_WIN, min_periods=ROLL_WIN//2).quantile(0.50)
    regime_map = (macro_series >= thresh).map({True:"Top", False:"Btm"})
    
    for regime in ["Top","Btm","Com"]:
        if regime == "Com":
            dates = regime_map.index.intersection(cr_c.index)
        else:
            dates = regime_map[regime_map == regime].index.intersection(cr_c.index)
        
        cr_sub = cr_c.loc[dates].dropna(axis=0, how="all")
        if len(cr_sub) < 24:
            rows.append({"Variable":paper_name,"Regime":regime,
                         "AVG":"","SR":"","GRS":"","p-value":"",
                         "mean|α|":"","√mean(α²)":"","mean R²":""})
            continue
        
        # Tangency portfolio return
        try:
            b, _ = estimate_hj_sdf(cr_sub, ridge=1e-3)
            tan_ret = cr_sub.values @ b
            tan_ser = pd.Series(tan_ret, index=cr_sub.index)
        except Exception:
            tan_ser = cr_sub.mean(axis=1)
        
        avg  = tan_ser.mean() * 100
        sr   = ann_sharpe(tan_ser)
        
        if regime == "Com":
            rows.append({"Variable":paper_name,"Regime":"Com",
                         "AVG":f"{avg:.2f}","SR":f"{sr:.2f}",
                         "GRS":"","p-value":"","mean|α|":"","√mean(α²)":"","mean R²":""})
        else:
            fd_sub = fd_c.loc[fd_c.index.intersection(cr_sub.index)]
            grs_f, p_val, alphas, r2s = grs_test(cr_sub, fd_sub, ff5_avail)
            
            alphas_arr = np.array([a for a in alphas if not np.isnan(a)])
            r2s_arr    = np.array([r for r in r2s    if not np.isnan(r)])
            
            rows.append({
                "Variable":    paper_name,
                "Regime":      regime,
                "AVG":         f"{avg:.2f}",
                "SR":          f"{sr:.2f}",
                "GRS":         f"{grs_f:.2f}" if not np.isnan(grs_f) else "",
                "p-value":     f"{p_val:.2f}" if not np.isnan(p_val) else "",
                "mean|α|":     f"{np.mean(np.abs(alphas_arr)):.2f}" if len(alphas_arr)>0 else "",
                "√mean(α²)":   f"{np.sqrt(np.mean(alphas_arr**2)):.2f}" if len(alphas_arr)>0 else "",
                "mean R²":     f"{np.mean(r2s_arr):.2f}" if len(r2s_arr)>0 else "",
            })
    print("done")

table8 = pd.DataFrame(rows).set_index(["Variable","Regime"])
print("\n=== Table 8 ===")
print(table8.to_string())
save_table(table8, "Table8_Macro_Regime_Switches")
print("\nTable 8 saved.")

NameError: name 'avail_macro' is not defined

NameError: name 'df' is not defined